# 第十八课:KV Cache——推理提速的第一杠杆

讲义:[18-kv-cache.md](18-kv-cache.md)。本课用第 20 课 Mini-GPT 的部件现场验证缓存等价性。

In [ ]:
import numpy as np, math
rng = np.random.default_rng(0)

def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

def causal_mask(n, T):
    return np.triu(np.ones((n, T), bool), T - n + 1)     # +1: 允许看到自己

# 单测:每行可见列数 = T-n+1+i
for n, T in [(3, 3), (1, 5), (2, 7)]:
    vis = (~causal_mask(n, T)).sum(-1)
    print(f'n={n} T={T} 可见列数={vis}')

In [ ]:
# 迷你模型(1 层,足以演示 KV Cache)
V, D, H = 16, 16, 2
DH = D // H
E = rng.normal(size=(V, D)) * 0.5
Wq, Wk, Wv, Wo = (rng.normal(size=(D, D)) * 0.1 for _ in range(4))
HEAD = rng.normal(size=(D, V)) * 0.1

def attn(x, pos, cache=None):
    n = x.shape[0]
    q, k, v = x @ Wq, x @ Wk, x @ Wv
    if cache is not None:
        k = np.concatenate([cache[0], k], 0)
        v = np.concatenate([cache[1], v], 0)
    outs = []
    for h in range(H):
        sl = slice(h*DH, (h+1)*DH)
        s = q[:, sl] @ k[:, sl].T / math.sqrt(DH)
        s = np.where(causal_mask(n, k.shape[0]), -np.inf, s)
        outs.append(softmax(s) @ v[:, sl])
    return np.concatenate(outs, -1) @ Wo, (k, v)

def forward(ids, cache=None):
    pos0 = cache[0][0].shape[0] if cache else 0
    x = E[ids]
    x, kv = attn(x, np.arange(pos0, pos0 + len(ids)), cache)
    logits = x / np.sqrt((x**2).mean(-1, keepdims=True) + 1e-5) @ HEAD   # RMSNorm + head
    return logits, kv

In [ ]:
# A) 无缓存:每步全量重算
ids = [1, 2, 3]
for _ in range(8):
    lg, _ = forward(ids)
    ids.append(int(np.argmax(lg[-1])))
print('无缓存:', ids)

# B) KV Cache:prefill + 逐步 decode
ids2 = [1, 2, 3]
lg, kv = forward(ids2)                      # prefill
for _ in range(8):
    nxt = int(np.argmax(lg[-1]))
    ids2.append(nxt)
    lg, kv = forward([nxt], kv)              # decode: 每步 1 个 token
print('缓存版:', ids2, ' 一致:', ids == ids2)

# C) 逐 token logits 对照(数学等价性)
seq = list(range(1, 12))
lg_full, _ = forward(seq)
cached = []
lg2, kv2 = forward(seq[:4])
for t in seq[4:]:
    lg2, kv2 = forward([t], kv2)
    cached.append(lg2[-1])
print('逐 token logits 最大误差:', float(np.abs(np.array(cached) - lg_full[4:]).max()))
assert np.isfinite(lg_full).all(), 'NaN 检查'

In [ ]:
# 显存账(作业 D1 的模板)
def kv_cache_bytes(n_layers, n_kv_heads, d_h, seq, batch=1, bytes_per=2):
    return 2 * n_layers * n_kv_heads * d_h * seq * batch * bytes_per

print('Llama-3-8B 8K bf16:', round(kv_cache_bytes(32, 8, 128, 8192)/1024**3, 2), 'GB')
print('同款 batch=32    :', round(kv_cache_bytes(32, 8, 128, 8192, 32)/1024**3, 1), 'GB')

In [ ]:
# TODO(E1): 计时对比:100 token 生成,无缓存 vs 缓存,报告加速比
# import time
# ... 两种循环各跑一次,print 时间比

# TODO(C2): 把误差断言改成 < 1e-10,验证你的实现